# Indonesian Emotion Detection - Kaggle Training Notebook

This notebook trains **5 models** for Indonesian emotion detection, optimized for Kaggle environment.

## 🎯 Models to Train
1. **Logistic Regression**: TF-IDF + LogReg with calibration
2. **Linear SVM**: TF-IDF + SVM with probability calibration
3. **Naive Bayes**: TF-IDF + Multinomial NB
4. **Bidirectional GRU**: Deep learning with word embeddings
5. **Transformers**: IndoBERT and RoBERTa fine-tuning

## 📊 Evaluation Metric
- **Primary**: F1 Macro Score
- **Secondary**: Accuracy, Per-class F1, Confusion Matrix

## 🚀 Kaggle Setup
- Enable GPU for faster training
- Enable internet access for model downloads
- Expected runtime: ~60-80 minutes

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets accelerate
!pip install scikit-learn pandas numpy matplotlib seaborn
!pip install joblib

print("✅ All dependencies installed successfully!")

In [ ]:
# Import all required libraries
import sys
import os
import time
import warnings
import pickle
import re
import gc
import joblib
from pathlib import Path
from typing import Any, Dict, List, Optional, Union, Tuple
from abc import ABC, abstractmethod
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.optim import Adam, AdamW
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)

# Suppress warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"🐍 Python version: {sys.version}")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🚀 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# Configuration - Based on configs/emotion.yaml
CONFIG = {
    # Data paths (adjust for Kaggle)
    'data_dir': '/kaggle/input/emotion-dataset-indonesian/Emotion Dataset from Indonesian Public Opinion',
    'output_dir': Path('/kaggle/working/artifacts'),
    
    # General settings
    'random_state': 42,
    'test_size': 0.2,
    'val_size': 0.1,  # From remaining train data
    
    # TF-IDF parameters (for sklearn models)
    'max_features': 20000,
    'ngram_min': 1,
    'ngram_max': 2,
    'min_df': 2,
    'max_df': 0.95,
    
    # Model-specific configurations
    'logreg': {
        'C': 1.0,
        'max_iter': 2000,
        'class_weight': 'balanced',
    },
    
    'svm': {
        'C': 1.0,
        'max_iter': 2000,
        'class_weight': 'balanced',
    },
    
    'nb': {
        'alpha': 1.0,
    },
    
    'bigru': {
        'vocab_size': 20000,
        'embedding_dim': 128,
        'hidden_dim': 128,
        'num_layers': 2,
        'dropout': 0.3,
        'max_length': 128,
        'epochs': 20,
        'batch_size': 32,
        'learning_rate': 0.001,
        'patience': 5,
    },
    
    'transformer': {
        'models': {
            'indobert': 'indobenchmark/indobert-base-p1',
            'roberta': 'indolem/indobert-base-uncased',  # Alternative Indonesian RoBERTa
        },
        'max_length': 256,
        'num_epochs': 3,
        'batch_size': 16,
        'learning_rate': 2e-5,
        'warmup_ratio': 0.06,
        'weight_decay': 0.01,
    }
}

# Set random seeds for reproducibility
np.random.seed(CONFIG['random_state'])
torch.manual_seed(CONFIG['random_state'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['random_state'])

# Create output directory
CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)

print("🔧 Configuration loaded:")
print(f"   Data directory: {CONFIG['data_dir']}")
print(f"   Output directory: {CONFIG['output_dir']}")
print(f"   Random seed: {CONFIG['random_state']}")
print(f"   GPU available: {torch.cuda.is_available()}")

## 3. Core Classes - Text Preprocessing

In [ ]:
# Indonesian Text Preprocessor
class IndonesianTextPreprocessor:
    """Text preprocessing for Indonesian language."""
    
    def __init__(self, lowercase: bool = True, remove_urls: bool = True,
                 remove_mentions: bool = True, remove_hashtags: bool = False,
                 min_length: int = 1) -> None:
        self.lowercase = lowercase
        self.remove_urls = remove_urls
        self.remove_mentions = remove_mentions
        self.remove_hashtags = remove_hashtags
        self.min_length = min_length
        
        # Regex patterns
        self.url_pattern = re.compile(r'https?://\S+|www\.\S+')
        self.mention_pattern = re.compile(r'@\w+')
        self.hashtag_pattern = re.compile(r'#(\w+)')
    
    def normalize_text(self, text: str) -> str:
        """Normalize text."""
        if not isinstance(text, str):
            return ""
        
        # Remove URLs
        if self.remove_urls:
            text = self.url_pattern.sub('', text)
        
        # Remove mentions
        if self.remove_mentions:
            text = self.mention_pattern.sub('', text)
        
        # Remove hashtags (keep the word)
        if self.remove_hashtags:
            text = self.hashtag_pattern.sub(r'\1', text)
        
        # Lowercase
        if self.lowercase:
            text = text.lower()
        
        # Normalize whitespace
        text = re.sub(r'[\r\n]+', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()
        
        return text
    
    def preprocess(self, text: str) -> str:
        """Preprocess a single text."""
        text = self.normalize_text(text)
        
        # Filter by minimum length
        tokens = text.split()
        tokens = [t for t in tokens if len(t) >= self.min_length]
        
        return ' '.join(tokens)
    
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        """Preprocess a batch of texts."""
        return [self.preprocess(text) for text in texts]

print("✅ Text preprocessor defined!")

## 4. Data Loading & Preprocessing

In [ ]:
# Load emotion dataset
def load_emotion_dataset(data_dir: str) -> pd.DataFrame:
    """Load emotion dataset from CSV files."""
    data_dir = Path(data_dir)
    
    # Emotion files
    emotion_files = {
        'anger': 'AngerData.csv',
        'fear': 'FearData.csv',
        'joy': 'JoyData.csv',
        'love': 'LoveData.csv',
        'neutral': 'NeutralData.csv',
        'sadness': 'SadData.csv',
    }
    
    all_data = []
    
    for emotion, filename in emotion_files.items():
        filepath = data_dir / filename
        
        if not filepath.exists():
            print(f"⚠️  Warning: {filename} not found, skipping...")
            continue
        
        df = pd.read_csv(filepath)
        
        # Auto-detect text column
        text_candidates = ['text', 'Text', 'tweet', 'Tweet', 'sentence', 'content', 'message']
        text_col = None
        for col in text_candidates:
            if col in df.columns:
                text_col = col
                break
        
        if text_col is None:
            print(f"⚠️  Warning: No text column found in {filename}, using first column")
            text_col = df.columns[0]
        
        # Create standardized dataframe
        emotion_df = pd.DataFrame({
            'text': df[text_col],
            'emotion': emotion
        })
        
        all_data.append(emotion_df)
        print(f"✅ Loaded {len(emotion_df)} samples from {filename} (emotion: {emotion})")
    
    # Concatenate all data
    dataset = pd.concat(all_data, ignore_index=True)
    
    # Clean data
    dataset = dataset.dropna(subset=['text', 'emotion'])
    dataset = dataset[dataset['text'].str.strip().str.len() > 0]
    dataset = dataset.drop_duplicates(subset=['text'])
    
    return dataset

print("✅ Data loading function defined!")

In [ ]:
# Load and prepare dataset
print("📊 Loading emotion dataset...\n")

dataset = load_emotion_dataset(CONFIG['data_dir'])

print(f"\n📈 Dataset statistics:")
print(f"   Total samples: {len(dataset):,}")
print(f"   Number of emotions: {dataset['emotion'].nunique()}")
print(f"\n   Emotion distribution:")
for emotion, count in dataset['emotion'].value_counts().items():
    print(f"      {emotion}: {count:,} ({count/len(dataset)*100:.1f}%)")

# Create label encoder
emotion_labels = sorted(dataset['emotion'].unique())
label_to_id = {label: idx for idx, label in enumerate(emotion_labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

dataset['label'] = dataset['emotion'].map(label_to_id)

print(f"\n🏷️  Label mapping:")
for emotion, idx in label_to_id.items():
    print(f"   {idx}: {emotion}")

# Split dataset
train_val_df, test_df = train_test_split(
    dataset,
    test_size=CONFIG['test_size'],
    random_state=CONFIG['random_state'],
    stratify=dataset['label']
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=CONFIG['val_size'],
    random_state=CONFIG['random_state'],
    stratify=train_val_df['label']
)

# Prepare data arrays
X_train = train_df['text'].tolist()
y_train = train_df['label'].values
X_val = val_df['text'].tolist()
y_val = val_df['label'].values
X_test = test_df['text'].tolist()
y_test = test_df['label'].values

print(f"\n📋 Data split:")
print(f"   Training: {len(X_train):,} samples")
print(f"   Validation: {len(X_val):,} samples")
print(f"   Test: {len(X_test):,} samples")

# Show sample texts
print(f"\n📝 Sample texts:")
for i in range(min(3, len(train_df))):
    text = X_train[i][:100] + '...' if len(X_train[i]) > 100 else X_train[i]
    emotion = id_to_label[y_train[i]]
    print(f"   [{emotion}] {text}")

## 5. Evaluation Metrics

In [ ]:
# Evaluation functions
def compute_f1_macro(y_true, y_pred):
    """Compute F1 Macro score."""
    return f1_score(y_true, y_pred, average='macro')

def evaluate_model(model_name: str, y_true, y_pred, training_time: float = None):
    """Evaluate model and return metrics."""
    
    # Compute metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_weighted = f1_score(y_true, y_pred, average='weighted')
    precision_macro = precision_score(y_true, y_pred, average='macro')
    recall_macro = recall_score(y_true, y_pred, average='macro')
    
    metrics = {
        'model': model_name,
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'training_time': training_time
    }
    
    print(f"\n📊 {model_name} Evaluation Results:")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1 Macro: {f1_macro:.4f} ⭐")
    print(f"   F1 Weighted: {f1_weighted:.4f}")
    print(f"   Precision Macro: {precision_macro:.4f}")
    print(f"   Recall Macro: {recall_macro:.4f}")
    if training_time:
        print(f"   Training Time: {training_time:.2f}s")
    
    return metrics

def plot_confusion_matrix(y_true, y_pred, labels, title="Confusion Matrix"):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                cbar_kws={'label': 'Count'})
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    return cm

def print_classification_report(y_true, y_pred, labels):
    """Print detailed classification report."""
    print("\n📋 Detailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=labels, digits=4))

print("✅ Evaluation functions defined!")

## 6. Model 1 - Logistic Regression

In [ ]:
# Logistic Regression Model
print("🚀 Training Logistic Regression model...\n")

start_time = time.time()

# Preprocessing
preprocessor = IndonesianTextPreprocessor(lowercase=True, remove_urls=True, remove_mentions=True)
X_train_clean = preprocessor.preprocess_batch(X_train)
X_val_clean = preprocessor.preprocess_batch(X_val)
X_test_clean = preprocessor.preprocess_batch(X_test)

# Build pipeline
logreg_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=CONFIG['max_features'],
        ngram_range=(CONFIG['ngram_min'], CONFIG['ngram_max']),
        min_df=CONFIG['min_df'],
        max_df=CONFIG['max_df'],
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        C=CONFIG['logreg']['C'],
        max_iter=CONFIG['logreg']['max_iter'],
        class_weight=CONFIG['logreg']['class_weight'],
        random_state=CONFIG['random_state'],
        n_jobs=-1
    ))
])

# Train
logreg_pipeline.fit(X_train_clean, y_train)

training_time = time.time() - start_time

# Evaluate
y_train_pred_lr = logreg_pipeline.predict(X_train_clean)
y_val_pred_lr = logreg_pipeline.predict(X_val_clean)
y_test_pred_lr = logreg_pipeline.predict(X_test_clean)

print("\n📊 Train Set:")
logreg_train_metrics = evaluate_model("LogReg", y_train, y_train_pred_lr, training_time)

print("\n📊 Validation Set:")
logreg_val_metrics = evaluate_model("LogReg", y_val, y_val_pred_lr)

print("\n📊 Test Set:")
logreg_test_metrics = evaluate_model("LogReg", y_test, y_test_pred_lr)

# Save model
logreg_path = CONFIG['output_dir'] / 'logreg'
logreg_path.mkdir(exist_ok=True)
joblib.dump(logreg_pipeline, logreg_path / 'model.pkl')
print(f"\n💾 Model saved to {logreg_path}")

## 7. Model 2 - Linear SVM

In [ ]:
# Linear SVM Model
print("🚀 Training Linear SVM model...\n")

start_time = time.time()

# Build pipeline with calibrated SVM
svm_base = LinearSVC(
    C=CONFIG['svm']['C'],
    max_iter=CONFIG['svm']['max_iter'],
    class_weight=CONFIG['svm']['class_weight'],
    random_state=CONFIG['random_state']
)

svm_calibrated = CalibratedClassifierCV(svm_base, cv=3)

svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=CONFIG['max_features'],
        ngram_range=(CONFIG['ngram_min'], CONFIG['ngram_max']),
        min_df=CONFIG['min_df'],
        max_df=CONFIG['max_df'],
        sublinear_tf=True
    )),
    ('clf', svm_calibrated)
])

# Train
svm_pipeline.fit(X_train_clean, y_train)

training_time = time.time() - start_time

# Evaluate
y_train_pred_svm = svm_pipeline.predict(X_train_clean)
y_val_pred_svm = svm_pipeline.predict(X_val_clean)
y_test_pred_svm = svm_pipeline.predict(X_test_clean)

print("\n📊 Train Set:")
svm_train_metrics = evaluate_model("SVM", y_train, y_train_pred_svm, training_time)

print("\n📊 Validation Set:")
svm_val_metrics = evaluate_model("SVM", y_val, y_val_pred_svm)

print("\n📊 Test Set:")
svm_test_metrics = evaluate_model("SVM", y_test, y_test_pred_svm)

# Save model
svm_path = CONFIG['output_dir'] / 'svm'
svm_path.mkdir(exist_ok=True)
joblib.dump(svm_pipeline, svm_path / 'model.pkl')
print(f"\n💾 Model saved to {svm_path}")

## 8. Model 3 - Naive Bayes

In [ ]:
# Naive Bayes Model
print("🚀 Training Naive Bayes model...\n")

start_time = time.time()

# Build pipeline
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=CONFIG['max_features'],
        ngram_range=(CONFIG['ngram_min'], CONFIG['ngram_max']),
        min_df=CONFIG['min_df'],
        max_df=CONFIG['max_df'],
        sublinear_tf=True
    )),
    ('clf', MultinomialNB(alpha=CONFIG['nb']['alpha']))
])

# Train
nb_pipeline.fit(X_train_clean, y_train)

training_time = time.time() - start_time

# Evaluate
y_train_pred_nb = nb_pipeline.predict(X_train_clean)
y_val_pred_nb = nb_pipeline.predict(X_val_clean)
y_test_pred_nb = nb_pipeline.predict(X_test_clean)

print("\n📊 Train Set:")
nb_train_metrics = evaluate_model("Naive Bayes", y_train, y_train_pred_nb, training_time)

print("\n📊 Validation Set:")
nb_val_metrics = evaluate_model("Naive Bayes", y_val, y_val_pred_nb)

print("\n📊 Test Set:")
nb_test_metrics = evaluate_model("Naive Bayes", y_test, y_test_pred_nb)

# Save model
nb_path = CONFIG['output_dir'] / 'nb'
nb_path.mkdir(exist_ok=True)
joblib.dump(nb_pipeline, nb_path / 'model.pkl')
print(f"\n💾 Model saved to {nb_path}")

## 9. Model 4 - Bidirectional GRU

In [ ]:
# BiGRU Model Implementation
class BiGRUClassifier(nn.Module):
    """Bidirectional GRU for emotion classification."""
    
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout, num_classes):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(
            embedding_dim, hidden_dim, num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    
    def forward(self, x):
        embedded = self.embedding(x)
        gru_out, hidden = self.gru(embedded)
        
        # Concatenate last hidden states from both directions
        hidden = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        hidden = self.dropout(hidden)
        
        output = self.fc(hidden)
        return output


class BiGRUModel:
    """Wrapper for BiGRU training and inference."""
    
    def __init__(self, config, num_classes):
        self.config = config
        self.num_classes = num_classes
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.preprocessor = IndonesianTextPreprocessor()
        
        self.word_to_idx = {'<PAD>': 0, '<UNK>': 1}
        self.model = None
    
    def build_vocab(self, texts):
        """Build vocabulary from texts."""
        word_freq = Counter()
        
        for text in texts:
            text_clean = self.preprocessor.preprocess(text)
            words = text_clean.split()
            word_freq.update(words)
        
        # Keep top vocab_size - 2 words (accounting for PAD and UNK)
        most_common = word_freq.most_common(self.config['vocab_size'] - 2)
        
        for word, _ in most_common:
            if word not in self.word_to_idx:
                self.word_to_idx[word] = len(self.word_to_idx)
        
        print(f"   Vocabulary size: {len(self.word_to_idx):,}")
    
    def texts_to_sequences(self, texts):
        """Convert texts to sequences."""
        sequences = []
        
        for text in texts:
            text_clean = self.preprocessor.preprocess(text)
            words = text_clean.split()[:self.config['max_length']]
            seq = [self.word_to_idx.get(word, 1) for word in words]  # 1 is UNK
            
            # Pad sequence
            if len(seq) < self.config['max_length']:
                seq = seq + [0] * (self.config['max_length'] - len(seq))
            
            sequences.append(seq)
        
        return torch.LongTensor(sequences)
    
    def train(self, X_train, y_train, X_val, y_val):
        """Train the BiGRU model."""
        print("   Building vocabulary...")
        self.build_vocab(X_train)
        
        print("   Converting texts to sequences...")
        X_train_seq = self.texts_to_sequences(X_train)
        X_val_seq = self.texts_to_sequences(X_val)
        y_train_tensor = torch.LongTensor(y_train)
        y_val_tensor = torch.LongTensor(y_val)
        
        # Create data loaders
        train_dataset = TensorDataset(X_train_seq, y_train_tensor)
        val_dataset = TensorDataset(X_val_seq, y_val_tensor)
        
        train_loader = DataLoader(
            train_dataset, batch_size=self.config['batch_size'], shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, batch_size=self.config['batch_size']
        )
        
        # Initialize model
        print("   Initializing model...")
        self.model = BiGRUClassifier(
            vocab_size=len(self.word_to_idx),
            embedding_dim=self.config['embedding_dim'],
            hidden_dim=self.config['hidden_dim'],
            num_layers=self.config['num_layers'],
            dropout=self.config['dropout'],
            num_classes=self.num_classes
        ).to(self.device)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = Adam(self.model.parameters(), lr=self.config['learning_rate'])
        
        # Training loop
        print(f"   Training for {self.config['epochs']} epochs...")
        best_val_f1 = 0
        patience_counter = 0
        
        for epoch in range(self.config['epochs']):
            self.model.train()
            train_loss = 0
            
            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)
                
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                
                train_loss += loss.item()
            
            # Validation
            self.model.eval()
            val_preds = []
            val_true = []
            
            with torch.no_grad():
                for batch_X, batch_y in val_loader:
                    batch_X = batch_X.to(self.device)
                    outputs = self.model(batch_X)
                    preds = torch.argmax(outputs, dim=1).cpu().numpy()
                    val_preds.extend(preds)
                    val_true.extend(batch_y.numpy())
            
            val_f1 = f1_score(val_true, val_preds, average='macro')
            
            if (epoch + 1) % 5 == 0:
                print(f"   Epoch {epoch+1}/{self.config['epochs']} - "
                      f"Loss: {train_loss/len(train_loader):.4f}, "
                      f"Val F1: {val_f1:.4f}")
            
            # Early stopping
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= self.config['patience']:
                print(f"   Early stopping at epoch {epoch+1}")
                break
    
    def predict(self, texts):
        """Predict emotions for texts."""
        self.model.eval()
        X_seq = self.texts_to_sequences(texts)
        
        dataset = TensorDataset(X_seq)
        loader = DataLoader(dataset, batch_size=self.config['batch_size'])
        
        predictions = []
        
        with torch.no_grad():
            for batch_X, in loader:
                batch_X = batch_X.to(self.device)
                outputs = self.model(batch_X)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                predictions.extend(preds)
        
        return np.array(predictions)
    
    def save(self, path):
        """Save model."""
        path = Path(path)
        path.mkdir(parents=True, exist_ok=True)
        
        torch.save(self.model.state_dict(), path / 'model.pth')
        joblib.dump({
            'word_to_idx': self.word_to_idx,
            'config': self.config,
            'num_classes': self.num_classes
        }, path / 'vocab.pkl')

print("✅ BiGRU model class defined!")

In [ ]:
# Train BiGRU Model
print("🚀 Training Bidirectional GRU model...\n")

start_time = time.time()

bigru_model = BiGRUModel(CONFIG['bigru'], num_classes=len(emotion_labels))
bigru_model.train(X_train, y_train, X_val, y_val)

training_time = time.time() - start_time

# Evaluate
print("\n   Evaluating on test set...")
y_train_pred_gru = bigru_model.predict(X_train)
y_val_pred_gru = bigru_model.predict(X_val)
y_test_pred_gru = bigru_model.predict(X_test)

print("\n📊 Train Set:")
bigru_train_metrics = evaluate_model("BiGRU", y_train, y_train_pred_gru, training_time)

print("\n📊 Validation Set:")
bigru_val_metrics = evaluate_model("BiGRU", y_val, y_val_pred_gru)

print("\n📊 Test Set:")
bigru_test_metrics = evaluate_model("BiGRU", y_test, y_test_pred_gru)

# Save model
bigru_path = CONFIG['output_dir'] / 'bigru'
bigru_model.save(bigru_path)
print(f"\n💾 Model saved to {bigru_path}")

# Clear GPU memory
del bigru_model
torch.cuda.empty_cache()
gc.collect()

## 10. Model 5 - Transformers (IndoBERT)

In [ ]:
# Transformer Dataset class
class EmotionDataset(Dataset):
    """Dataset for transformer models."""
    
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)


def train_transformer_model(model_name, model_key, X_train, y_train, X_val, y_val, X_test, y_test):
    """Train a transformer model."""
    
    print(f"\n🚀 Training {model_key.upper()} model...\n")
    print(f"   Model: {model_name}")
    
    start_time = time.time()
    
    # Load tokenizer and model
    print("   Loading tokenizer and model...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(emotion_labels)
    )
    
    # Tokenize
    print("   Tokenizing texts...")
    train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=CONFIG['transformer']['max_length'])
    val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=CONFIG['transformer']['max_length'])
    test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=CONFIG['transformer']['max_length'])
    
    # Create datasets
    train_dataset = EmotionDataset(train_encodings, y_train)
    val_dataset = EmotionDataset(val_encodings, y_val)
    test_dataset = EmotionDataset(test_encodings, y_test)
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=str(CONFIG['output_dir'] / f'{model_key}_checkpoints'),
        num_train_epochs=CONFIG['transformer']['num_epochs'],
        per_device_train_batch_size=CONFIG['transformer']['batch_size'],
        per_device_eval_batch_size=CONFIG['transformer']['batch_size'],
        learning_rate=CONFIG['transformer']['learning_rate'],
        warmup_ratio=CONFIG['transformer']['warmup_ratio'],
        weight_decay=CONFIG['transformer']['weight_decay'],
        logging_dir=str(CONFIG['output_dir'] / f'{model_key}_logs'),
        logging_steps=100,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        save_total_limit=2,
        seed=CONFIG['random_state'],
    )
    
    # Metrics function
    def compute_metrics(pred):
        labels = pred.label_ids
        preds = pred.predictions.argmax(-1)
        
        f1_macro = f1_score(labels, preds, average='macro')
        f1_weighted = f1_score(labels, preds, average='weighted')
        accuracy = accuracy_score(labels, preds)
        
        return {
            'f1_macro': f1_macro,
            'f1_weighted': f1_weighted,
            'accuracy': accuracy
        }
    
    # Trainer
    print("   Training...")
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
    
    trainer.train()
    
    training_time = time.time() - start_time
    
    # Evaluate
    print("\n   Evaluating...")
    train_pred = trainer.predict(train_dataset)
    val_pred = trainer.predict(val_dataset)
    test_pred = trainer.predict(test_dataset)
    
    y_train_pred = train_pred.predictions.argmax(-1)
    y_val_pred = val_pred.predictions.argmax(-1)
    y_test_pred = test_pred.predictions.argmax(-1)
    
    print(f"\n📊 {model_key.upper()} Train Set:")
    train_metrics = evaluate_model(model_key.upper(), y_train, y_train_pred, training_time)
    
    print(f"\n📊 {model_key.upper()} Validation Set:")
    val_metrics = evaluate_model(model_key.upper(), y_val, y_val_pred)
    
    print(f"\n📊 {model_key.upper()} Test Set:")
    test_metrics = evaluate_model(model_key.upper(), y_test, y_test_pred)
    
    # Save model
    model_path = CONFIG['output_dir'] / model_key
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"\n💾 Model saved to {model_path}")
    
    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()
    
    return train_metrics, val_metrics, test_metrics, y_train_pred, y_val_pred, y_test_pred

print("✅ Transformer training function defined!")

In [ ]:
# Train IndoBERT
indobert_train_metrics, indobert_val_metrics, indobert_test_metrics, \
y_train_pred_indobert, y_val_pred_indobert, y_test_pred_indobert = train_transformer_model(
    CONFIG['transformer']['models']['indobert'],
    'indobert',
    X_train, y_train, X_val, y_val, X_test, y_test
)

## 11. Model 6 - Transformers (RoBERTa)

In [ ]:
# Train RoBERTa
roberta_train_metrics, roberta_val_metrics, roberta_test_metrics, \
y_train_pred_roberta, y_val_pred_roberta, y_test_pred_roberta = train_transformer_model(
    CONFIG['transformer']['models']['roberta'],
    'roberta',
    X_train, y_train, X_val, y_val, X_test, y_test
)

## 12. Model Comparison

In [ ]:
# Create comparison dataframe
comparison_results = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'Type': 'Sklearn',
        'Train F1 Macro': logreg_train_metrics['f1_macro'],
        'Val F1 Macro': logreg_val_metrics['f1_macro'],
        'Test F1 Macro': logreg_test_metrics['f1_macro'],
        'Test Accuracy': logreg_test_metrics['accuracy'],
        'Training Time (s)': logreg_train_metrics['training_time']
    },
    {
        'Model': 'Linear SVM',
        'Type': 'Sklearn',
        'Train F1 Macro': svm_train_metrics['f1_macro'],
        'Val F1 Macro': svm_val_metrics['f1_macro'],
        'Test F1 Macro': svm_test_metrics['f1_macro'],
        'Test Accuracy': svm_test_metrics['accuracy'],
        'Training Time (s)': svm_train_metrics['training_time']
    },
    {
        'Model': 'Naive Bayes',
        'Type': 'Sklearn',
        'Train F1 Macro': nb_train_metrics['f1_macro'],
        'Val F1 Macro': nb_val_metrics['f1_macro'],
        'Test F1 Macro': nb_test_metrics['f1_macro'],
        'Test Accuracy': nb_test_metrics['accuracy'],
        'Training Time (s)': nb_train_metrics['training_time']
    },
    {
        'Model': 'BiGRU',
        'Type': 'Deep Learning',
        'Train F1 Macro': bigru_train_metrics['f1_macro'],
        'Val F1 Macro': bigru_val_metrics['f1_macro'],
        'Test F1 Macro': bigru_test_metrics['f1_macro'],
        'Test Accuracy': bigru_test_metrics['accuracy'],
        'Training Time (s)': bigru_train_metrics['training_time']
    },
    {
        'Model': 'IndoBERT',
        'Type': 'Transformer',
        'Train F1 Macro': indobert_train_metrics['f1_macro'],
        'Val F1 Macro': indobert_val_metrics['f1_macro'],
        'Test F1 Macro': indobert_test_metrics['f1_macro'],
        'Test Accuracy': indobert_test_metrics['accuracy'],
        'Training Time (s)': indobert_train_metrics['training_time']
    },
    {
        'Model': 'RoBERTa',
        'Type': 'Transformer',
        'Train F1 Macro': roberta_train_metrics['f1_macro'],
        'Val F1 Macro': roberta_val_metrics['f1_macro'],
        'Test F1 Macro': roberta_test_metrics['f1_macro'],
        'Test Accuracy': roberta_test_metrics['accuracy'],
        'Training Time (s)': roberta_train_metrics['training_time']
    },
])

# Sort by Test F1 Macro
comparison_results = comparison_results.sort_values('Test F1 Macro', ascending=False)

print("\n" + "="*80)
print("🏆 MODEL COMPARISON RESULTS (Sorted by Test F1 Macro)")
print("="*80)
print(comparison_results.to_string(index=False, float_format='%.4f'))
print("="*80)

# Highlight best model
best_model_idx = comparison_results['Test F1 Macro'].idxmax()
best_model = comparison_results.loc[best_model_idx]
print(f"\n🥇 Best Model: {best_model['Model']}")
print(f"   Test F1 Macro: {best_model['Test F1 Macro']:.4f}")
print(f"   Test Accuracy: {best_model['Test Accuracy']:.4f}")
print(f"   Training Time: {best_model['Training Time (s)']:.2f}s")

# Save comparison results
comparison_results.to_csv(CONFIG['output_dir'] / 'model_comparison.csv', index=False)
print(f"\n💾 Comparison results saved to {CONFIG['output_dir'] / 'model_comparison.csv'}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# F1 Macro scores
ax1 = axes[0, 0]
x_pos = np.arange(len(comparison_results))
ax1.bar(x_pos, comparison_results['Test F1 Macro'], color='skyblue', alpha=0.8, edgecolor='black')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(comparison_results['Model'], rotation=45, ha='right')
ax1.set_ylabel('F1 Macro Score')
ax1.set_title('Test F1 Macro Score by Model')
ax1.grid(axis='y', alpha=0.3)

for i, v in enumerate(comparison_results['Test F1 Macro']):
    ax1.text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=9)

# Accuracy
ax2 = axes[0, 1]
ax2.bar(x_pos, comparison_results['Test Accuracy'], color='lightcoral', alpha=0.8, edgecolor='black')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(comparison_results['Model'], rotation=45, ha='right')
ax2.set_ylabel('Accuracy')
ax2.set_title('Test Accuracy by Model')
ax2.grid(axis='y', alpha=0.3)

for i, v in enumerate(comparison_results['Test Accuracy']):
    ax2.text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=9)

# Training time
ax3 = axes[1, 0]
ax3.bar(x_pos, comparison_results['Training Time (s)'], color='lightgreen', alpha=0.8, edgecolor='black')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(comparison_results['Model'], rotation=45, ha='right')
ax3.set_ylabel('Training Time (seconds)')
ax3.set_title('Training Time by Model')
ax3.set_yscale('log')
ax3.grid(axis='y', alpha=0.3)

for i, v in enumerate(comparison_results['Training Time (s)']):
    ax3.text(i, v * 1.2, f'{v:.1f}s', ha='center', va='bottom', fontsize=9)

# Train vs Val vs Test F1
ax4 = axes[1, 1]
x_pos = np.arange(len(comparison_results))
width = 0.25

ax4.bar(x_pos - width, comparison_results['Train F1 Macro'], width, label='Train', alpha=0.8)
ax4.bar(x_pos, comparison_results['Val F1 Macro'], width, label='Validation', alpha=0.8)
ax4.bar(x_pos + width, comparison_results['Test F1 Macro'], width, label='Test', alpha=0.8)

ax4.set_xticks(x_pos)
ax4.set_xticklabels(comparison_results['Model'], rotation=45, ha='right')
ax4.set_ylabel('F1 Macro Score')
ax4.set_title('F1 Macro: Train vs Val vs Test')
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n💾 Comparison plot saved to {CONFIG['output_dir'] / 'model_comparison.png'}")

## 13. Confusion Matrices

In [ ]:
# Plot confusion matrices for all models
models_predictions = [
    ('Logistic Regression', y_test_pred_lr),
    ('Linear SVM', y_test_pred_svm),
    ('Naive Bayes', y_test_pred_nb),
    ('BiGRU', y_test_pred_gru),
    ('IndoBERT', y_test_pred_indobert),
    ('RoBERTa', y_test_pred_roberta),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, (model_name, y_pred) in enumerate(models_predictions):
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=emotion_labels, yticklabels=emotion_labels,
                ax=axes[idx], cbar_kws={'label': 'Count'})
    
    axes[idx].set_title(f'{model_name}\n(F1 Macro: {comparison_results[comparison_results["Model"] == model_name]["Test F1 Macro"].values[0]:.4f})')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"💾 Confusion matrices saved to {CONFIG['output_dir'] / 'confusion_matrices.png'}")

## 14. Per-Class Performance Analysis

In [ ]:
# Detailed classification reports
print("\n" + "="*80)
print("📋 DETAILED CLASSIFICATION REPORTS")
print("="*80)

for model_name, y_pred in models_predictions:
    print(f"\n{model_name}:")
    print("-" * 80)
    print(classification_report(y_test, y_pred, target_names=emotion_labels, digits=4))

In [ ]:
# Per-class F1 scores visualization
per_class_f1 = {}

for model_name, y_pred in models_predictions:
    f1_scores = f1_score(y_test, y_pred, average=None)
    per_class_f1[model_name] = f1_scores

# Create dataframe
per_class_df = pd.DataFrame(per_class_f1, index=emotion_labels).T

print("\n📊 Per-Class F1 Scores:")
print(per_class_df.to_string(float_format='%.4f'))

# Visualize
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(emotion_labels))
width = 0.13

for i, (model_name, _) in enumerate(models_predictions):
    offset = (i - 2.5) * width
    ax.bar(x + offset, per_class_df.loc[model_name], width, label=model_name, alpha=0.8)

ax.set_xlabel('Emotion')
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Scores by Model')
ax.set_xticks(x)
ax.set_xticklabels(emotion_labels)
ax.legend(loc='best')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(CONFIG['output_dir'] / 'per_class_f1.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n💾 Per-class F1 plot saved to {CONFIG['output_dir'] / 'per_class_f1.png'}")

# Save per-class results
per_class_df.to_csv(CONFIG['output_dir'] / 'per_class_f1.csv')
print(f"💾 Per-class F1 data saved to {CONFIG['output_dir'] / 'per_class_f1.csv'}")

## 15. Hyperparameter Tuning Examples

In [ ]:
# Hyperparameter tuning example for Logistic Regression
print("🔧 Hyperparameter Tuning Example - Logistic Regression\n")
print("This example shows how to perform grid search for sklearn models.")
print("Uncomment and run to perform actual tuning (may take time).\n")

# Example code (commented out to save time)
tuning_example = '''
# Define parameter grid
param_grid = {
    'tfidf__max_features': [10000, 20000, 30000],
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'clf__C': [0.1, 1.0, 10.0],
}

# Create base pipeline
base_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1))
])

# Grid search with F1 macro as scoring metric
grid_search = GridSearchCV(
    base_pipeline,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2
)

# Fit
grid_search.fit(X_train_clean, y_train)

# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best F1 Macro:", grid_search.best_score_)

# Evaluate on test set
y_test_pred = grid_search.best_estimator_.predict(X_test_clean)
test_f1 = f1_score(y_test, y_test_pred, average='macro')
print("Test F1 Macro:", test_f1)
'''

print(tuning_example)

print("\n" + "="*80)
print("Tips for Hyperparameter Tuning:")
print("="*80)
print("\n1. Sklearn Models (LogReg, SVM, NB):")
print("   - Use GridSearchCV or RandomizedSearchCV")
print("   - Tune TF-IDF parameters: max_features, ngram_range, min_df, max_df")
print("   - Tune classifier parameters: C (LogReg, SVM), alpha (NB)")
print("   - Use 'f1_macro' as scoring metric")

print("\n2. BiGRU Model:")
print("   - Tune: embedding_dim, hidden_dim, num_layers, dropout")
print("   - Tune: learning_rate, batch_size, epochs")
print("   - Use validation set for early stopping")
print("   - Monitor validation F1 macro")

print("\n3. Transformer Models (IndoBERT, RoBERTa):")
print("   - Tune: learning_rate (try 1e-5, 2e-5, 3e-5, 5e-5)")
print("   - Tune: batch_size (8, 16, 32)")
print("   - Tune: warmup_ratio (0.0, 0.06, 0.1)")
print("   - Tune: weight_decay (0.0, 0.01, 0.1)")
print("   - Use early stopping with patience=2-3")
print("   - Monitor validation F1 macro")

print("\n4. General Tips:")
print("   - Always use F1 Macro as primary metric for multi-class classification")
print("   - Use stratified splits to maintain class balance")
print("   - Monitor both train and validation metrics to detect overfitting")
print("   - Save best model based on validation F1 macro")
print("   - Test final model only once on test set")

## 16. Final Summary & Export

In [ ]:
# Final summary
print("\n" + "="*80)
print("🎯 TRAINING COMPLETE - FINAL SUMMARY")
print("="*80)

print(f"\n📊 Dataset:")
print(f"   Total samples: {len(dataset):,}")
print(f"   Emotions: {', '.join(emotion_labels)}")
print(f"   Train/Val/Test split: {len(X_train):,}/{len(X_val):,}/{len(X_test):,}")

print(f"\n🏆 Best Model (by Test F1 Macro):")
print(f"   Model: {best_model['Model']}")
print(f"   Test F1 Macro: {best_model['Test F1 Macro']:.4f}")
print(f"   Test Accuracy: {best_model['Test Accuracy']:.4f}")
print(f"   Training Time: {best_model['Training Time (s)']:.2f}s")

print(f"\n📈 All Models (sorted by Test F1 Macro):")
for _, row in comparison_results.iterrows():
    print(f"   {row['Model']:20s} - F1 Macro: {row['Test F1 Macro']:.4f}, "
          f"Accuracy: {row['Test Accuracy']:.4f}, "
          f"Time: {row['Training Time (s)']:6.1f}s")

print(f"\n💾 Saved Artifacts:")
print(f"   Output directory: {CONFIG['output_dir']}")
print(f"   Models:")
for model_dir in ['logreg', 'svm', 'nb', 'bigru', 'indobert', 'roberta']:
    path = CONFIG['output_dir'] / model_dir
    if path.exists():
        print(f"      - {model_dir}/")

print(f"   Results:")
print(f"      - model_comparison.csv")
print(f"      - model_comparison.png")
print(f"      - confusion_matrices.png")
print(f"      - per_class_f1.csv")
print(f"      - per_class_f1.png")

print(f"\n🚀 Next Steps:")
print(f"   1. Download models and results from Kaggle output")
print(f"   2. Use the best model for production deployment")
print(f"   3. Monitor performance on new data")
print(f"   4. Retrain periodically with updated data")
print(f"   5. Consider ensemble methods combining top models")

print("\n" + "="*80)
print("✅ ALL TRAINING AND EVALUATION COMPLETE!")
print("="*80)

In [ ]:
# Save final training metadata
metadata = {
    'config': CONFIG,
    'emotion_labels': emotion_labels,
    'label_to_id': label_to_id,
    'id_to_label': id_to_label,
    'dataset_stats': {
        'total_samples': len(dataset),
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test),
        'num_classes': len(emotion_labels),
    },
    'best_model': {
        'name': best_model['Model'],
        'test_f1_macro': float(best_model['Test F1 Macro']),
        'test_accuracy': float(best_model['Test Accuracy']),
        'training_time': float(best_model['Training Time (s)']),
    },
    'comparison_results': comparison_results.to_dict('records'),
}

joblib.dump(metadata, CONFIG['output_dir'] / 'training_metadata.pkl')
print(f"💾 Training metadata saved to {CONFIG['output_dir'] / 'training_metadata.pkl'}")

# Also save as JSON for easy reading
import json

# Convert non-serializable objects
metadata_json = metadata.copy()
metadata_json['config']['data_dir'] = str(metadata_json['config']['data_dir'])
metadata_json['config']['output_dir'] = str(metadata_json['config']['output_dir'])

with open(CONFIG['output_dir'] / 'training_metadata.json', 'w') as f:
    json.dump(metadata_json, f, indent=2)

print(f"💾 Training metadata saved to {CONFIG['output_dir'] / 'training_metadata.json'}")
print("\n✅ All artifacts saved successfully!")